# Customer Complaint Classification — Phase 1 & 2 Starter Notebook
### "Is Reasoning Worth the Cost?" Mini Project

This notebook covers:
1. Setting up API access to an LLM
2. Testing your first API call
3. Loading and exploring the complaint dataset
4. Cleaning and preparing a balanced sample
5. Building the baseline **zero-shot** classification pipeline
6. Testing it and logging results

**How to use this notebook:** Run each cell in order, top to bottom (click the ▶ button on the left of each cell, or press Shift+Enter). Read the markdown notes above each code cell before running it.


## Step 0 — Install required packages
Run this once per Colab session.

In [ ]:
!pip install openai pandas scikit-learn -q
print("Packages installed.")

## Step 1 — Set up your FREE API key (Groq)

We're using the **free Groq API** — no credit card, no verification hoops. Groq runs open models (like Llama 3.3) at very high speed, with a generous free daily limit (thousands of requests/day) — plenty for this project.

1. Go to **console.groq.com**, sign in (Google login works).
2. Go to **API Keys** → **Create API Key**.
3. Copy the key and paste it below.

**Important:** the key must be wrapped in quote marks in the code, like `API_KEY = "your-key-here"` — without the quotes, Python tries to run it as code instead of treating it as text, which causes a `NameError`.

**Never commit your real API key to GitHub.** For now, paste it directly below (fine for solo testing). Later, use Colab's "Secrets" manager (🔑 icon in the left sidebar) so it's not visible in the notebook itself.


In [ ]:
from openai import OpenAI

# Option A (quick, for now): paste your key directly, WITH the quote marks around it
API_KEY = "PASTE_YOUR_GROQ_API_KEY_HERE"   # <-- keep the quotes! this must be a text string

# Option B (safer, recommended once it's working): use Colab Secrets
# from google.colab import userdata
# API_KEY = userdata.get('GROQ_API_KEY')

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=API_KEY,
)
MODEL_NAME = "openai/gpt-oss-120b"
print("Client ready.")

## Step 2 — Test call: make sure everything works end to end

In [ ]:
test_complaint = "I was charged twice for my subscription this month and support has not responded."
categories = ["Billing", "Product Defect", "Customer Service", "Fraud", "Other"]

prompt = f'Classify this customer complaint into exactly one of these categories: {categories}.\n\nComplaint: "{test_complaint}"\n\nReply with ONLY the category name, nothing else.'

response = client.chat.completions.create(
    model=MODEL_NAME,
    max_tokens=200,                 # raised: reasoning models need budget for thinking + answer
    reasoning_effort="low",         # keep thinking minimal for a simple classification task
    messages=[{"role": "user", "content": prompt}]
)
print("Model said:", response.choices[0].message.content.strip())

**Checkpoint:** if you saw a category printed above (like `Billing`), your API setup works. Don't move on until this works for everyone on the team.

## Step 3 — Load the dataset

1. Download the dataset from Kaggle: search **"Consumer Complaints Dataset for NLP"** (CFPB-based).
2. In Colab, click the folder icon on the left sidebar → upload icon → upload the CSV file.
3. Update the filename below to match exactly what you uploaded.

The CFPB dataset usually has columns like `Product` (the category) and `Consumer complaint narrative` (the complaint text) — if your column names differ, adjust the `TEXT_COL` and `LABEL_COL` variables below after checking `df.columns`.


In [ ]:
import pandas as pd

FILENAME = "complaints.csv"   # <-- change this to your uploaded file's name

df = pd.read_csv(FILENAME, low_memory=False)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head(3)

## Step 4 — Explore the data
Check category names and how balanced they are.

In [ ]:
TEXT_COL = "Consumer complaint narrative"   # <-- update if different
LABEL_COL = "Product"                          # <-- update if different

print(df[LABEL_COL].value_counts().head(20))

## Step 5 — Clean and narrow down to categories

CFPB data often has many overlapping product categories (e.g., "Credit card", "Credit card or prepaid card"). Pick **5 clear, distinct categories** for your project (matching the plan) and map similar raw labels into them.

Edit the `CATEGORY_MAP` dictionary below based on what you saw in Step 4's output — the keys are the RAW category names from the dataset, and the values are your 5 clean category names.


In [ ]:
# EXAMPLE — replace the keys on the left with whatever raw category names you actually see in your dataset
CATEGORY_MAP = {
    "Credit card or prepaid card": "Billing",
    "Credit card": "Billing",
    "Debt collection": "Customer Service",
    "Mortgage": "Product Defect",
    "Checking or savings account": "Fraud",
    "Student loan": "Other",
    # ... add more mappings as needed based on df[LABEL_COL].value_counts()
}

df["clean_category"] = df[LABEL_COL].map(CATEGORY_MAP)
df_clean = df.dropna(subset=["clean_category", TEXT_COL]).copy()

# Remove empty or extremely short complaints, and keep it manageable (< 500 characters, matches prior work)
df_clean = df_clean[df_clean[TEXT_COL].str.len().between(20, 500)]

print("Rows after cleaning:", len(df_clean))
print(df_clean["clean_category"].value_counts())

## Step 6 — Build a balanced sample and split into two sets

- **Prompt-design set** (~20 examples): used to build/refine your prompts. You'll look at these closely.
- **Test set** (the rest, e.g. 200 per category = 1000 total): used ONLY for final results. Don't peek at this while designing prompts — that would bias your results.


In [ ]:
PER_CATEGORY_DESIGN = 4      # ~20 total across 5 categories
PER_CATEGORY_TEST = 60       # adjust based on budget/time; 60 x 5 = 300 total test cases

design_rows, test_rows = [], []

for cat, group in df_clean.groupby("clean_category"):
    group = group.sample(frac=1, random_state=42)  # shuffle
    design_rows.append(group.iloc[:PER_CATEGORY_DESIGN])
    remaining = group.iloc[PER_CATEGORY_DESIGN:]
    test_rows.append(remaining.iloc[:PER_CATEGORY_TEST])

design_df = pd.concat(design_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)

print("Prompt-design set size:", len(design_df))
print("Test set size:", len(test_df))

design_df.to_csv("prompt_design_set.csv", index=False)
test_df.to_csv("test_set.csv", index=False)
print("\nSaved: prompt_design_set.csv and test_set.csv")
print("Download these from the Colab file browser and add them to your GitHub repo's /data folder.")

## Step 7 — Build the baseline zero-shot classification function

This is the simplest possible version: one plain instruction, no examples, no reasoning steps. This becomes your **baseline** — everything else (few-shot, CoT, self-consistency) will be compared against this.


In [ ]:
import time

CATEGORIES = sorted(df_clean["clean_category"].unique().tolist())

def classify_zero_shot(complaint_text):
    prompt = (
        f"Classify this customer complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"Reply with ONLY the category name, nothing else."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=200,              # raised: reasoning models need budget for thinking + answer
        reasoning_effort="low",      # keep thinking minimal for a simple classification task
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    prediction = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens
    return prediction, elapsed, tokens_used

# Quick test on one example
sample_text = design_df.iloc[0][TEXT_COL]
true_label = design_df.iloc[0]["clean_category"]
pred, t, tok = classify_zero_shot(sample_text)
print("Complaint:", sample_text[:150], "...")
print("True label:", true_label)
print("Predicted:", pred, "| Time:", round(t,2), "s | Tokens:", tok)

## Step 8 — Run the baseline on the prompt-design set and check accuracy

This is just a small sanity check (20 examples) before running the full experiment later. If accuracy looks reasonable here (not near-random), your pipeline is working correctly.


In [ ]:
results = []

for idx, row in design_df.iterrows():
    pred, elapsed, tokens = classify_zero_shot(row[TEXT_COL])
    results.append({
        "complaint": row[TEXT_COL],
        "true_label": row["clean_category"],
        "predicted": pred,
        "correct": pred.strip().lower() == row["clean_category"].strip().lower(),
        "time_sec": elapsed,
        "tokens": tokens,
    })
    print(f"[{idx+1}/{len(design_df)}] True: {row['clean_category']:20s} | Predicted: {pred}")

results_df = pd.DataFrame(results)
accuracy = results_df["correct"].mean()
print(f"\nZero-shot accuracy on prompt-design set: {accuracy:.2%}")
print(f"Average time per call: {results_df['time_sec'].mean():.2f}s")
print(f"Average tokens per call: {results_df['tokens'].mean():.1f}")

results_df.to_csv("baseline_zero_shot_design_results.csv", index=False)
print("\nSaved: baseline_zero_shot_design_results.csv")

## ✅ Checkpoint — What you've accomplished

- Confirmed API access works
- Loaded and cleaned the real dataset
- Built a balanced prompt-design set and test set
- Built and tested your first working classification function (zero-shot baseline)
- Got a first accuracy number

## Next steps (Phase 3)

Next, we'll build the other 3 prompting conditions on top of this same structure:
- **Few-shot** (add solved examples before the question)
- **Chain-of-Thought** (ask the model to explain its reasoning before answering)
- **Self-consistency** (ask multiple times, take the majority vote)

Come back with this notebook once Step 8 is working and showing a reasonable accuracy number, and we'll build those next together.


---
# Phase 3 — Building the Other 3 Prompting Conditions

Your zero-shot baseline is working (85% on the design set). Now let's add the other 3 conditions: **few-shot**, **chain-of-thought (CoT)**, and **self-consistency**. Each builds on the same pattern as `classify_zero_shot`.


## Few-shot: show the model a few solved examples first

We'll hand-pick 1 example per category from the prompt-design set to use as the "teaching examples," then classify everything else (never reusing a complaint as both an example and a test case).


In [ ]:
# Pick one example per category from the design set to use as few-shot examples
FEW_SHOT_EXAMPLES = design_df.groupby("clean_category").first().reset_index()
print(FEW_SHOT_EXAMPLES[["clean_category", TEXT_COL]])

In [ ]:
def build_few_shot_block():
    lines = []
    for _, row in FEW_SHOT_EXAMPLES.iterrows():
        lines.append(f'Complaint: "{row[TEXT_COL]}"\nCategory: {row["clean_category"]}\n')
    return "\n".join(lines)

FEW_SHOT_BLOCK = build_few_shot_block()

def classify_few_shot(complaint_text):
    prompt = (
        f"Here are some examples of customer complaints and their correct category:\n\n"
        f"{FEW_SHOT_BLOCK}\n"
        f"Now classify this new complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"Reply with ONLY the category name, nothing else."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=200,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    prediction = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens
    return prediction, elapsed, tokens_used

# Quick test
pred, t, tok = classify_few_shot(design_df.iloc[10][TEXT_COL])
print("Predicted:", pred, "| True:", design_df.iloc[10]["clean_category"], "| Time:", round(t,2), "| Tokens:", tok)

## Chain-of-Thought (CoT): ask the model to explain its reasoning first

This is also where we capture the model's **explanation** — you'll need this later for the faithfulness check (Phase 5).


In [ ]:
def classify_cot(complaint_text):
    prompt = (
        f"Classify this customer complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"First, think step by step about which category fits best and briefly explain your reasoning in 1-2 sentences. "
        f"Then, on a new final line, write EXACTLY: 'Final Category: <category name>'."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=300,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    full_text = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens

    # Extract the final category from the last line
    prediction = None
    for line in full_text.splitlines():
        if line.strip().lower().startswith("final category:"):
            prediction = line.split(":", 1)[1].strip()
    if prediction is None:
        prediction = full_text.strip().splitlines()[-1]  # fallback: last line

    explanation = full_text  # keep the full text for the faithfulness check later
    return prediction, explanation, elapsed, tokens_used

# Quick test
pred, expl, t, tok = classify_cot(design_df.iloc[10][TEXT_COL])
print("Predicted:", pred, "| True:", design_df.iloc[10]["clean_category"])
print("Explanation:", expl)
print("Time:", round(t,2), "| Tokens:", tok)

## Self-consistency: ask multiple times, take the majority vote

This calls `classify_cot` several times (default 5) for the same complaint and picks the most common final answer. This is the most expensive condition — it costs roughly N times as much as a single CoT call.


In [ ]:
from collections import Counter

def classify_self_consistency(complaint_text, n_samples=5):
    predictions = []
    explanations = []
    total_time = 0
    total_tokens = 0

    for _ in range(n_samples):
        pred, expl, t, tok = classify_cot(complaint_text)
        predictions.append(pred)
        explanations.append(expl)
        total_time += t
        total_tokens += tok

    vote_counts = Counter(predictions)
    majority_prediction = vote_counts.most_common(1)[0][0]

    return majority_prediction, explanations, total_time, total_tokens, vote_counts

# Quick test (this takes a few seconds since it makes 5 calls)
pred, expls, t, tok, votes = classify_self_consistency(design_df.iloc[10][TEXT_COL], n_samples=5)
print("Majority vote:", pred, "| True:", design_df.iloc[10]["clean_category"])
print("Vote breakdown:", dict(votes))
print("Total time:", round(t,2), "| Total tokens:", tok)

## ✅ Checkpoint — All 4 conditions built

You now have 4 working functions:
- `classify_zero_shot(text)` → prediction, time, tokens
- `classify_few_shot(text)` → prediction, time, tokens
- `classify_cot(text)` → prediction, explanation, time, tokens
- `classify_self_consistency(text, n_samples)` → majority prediction, explanations, total time, total tokens, vote breakdown

## Next: Phase 4 — run all 4 conditions on the full test set

Before running on all 300 test cases (expensive — self-consistency alone makes 5×300 = 1500 calls), first test all 4 functions on just the 20-example prompt-design set to sanity check accuracy and catch any bugs cheaply. Come back once you've run these quick tests and we'll build the full experiment-logging loop for Phase 4 together.


---
# Phase 4 — Full Experiment: Run All 4 Conditions on Every Complaint

This is the big run. We'll build one function that runs all 4 conditions on a single complaint and returns everything we need, then loop it over every row in a dataframe, saving progress as we go (so a crash halfway through doesn't lose everything).

**Cost/time expectation:** self-consistency makes 5 calls per complaint, so for 175 test complaints, total calls = 175 (zero-shot) + 175 (few-shot) + 175 (CoT) + 175×5 (self-consistency) = **1,400 API calls**. At Groq's speed this is roughly 15-25 minutes, well within the free daily limit. Run it once and let it finish — don't re-run from scratch if it's slow, just wait.


In [ ]:
import time

def run_all_conditions(complaint_text, true_label, sc_samples=5):
    """Runs all 4 prompting conditions on one complaint and returns a dict of results."""
    row_result = {"complaint": complaint_text, "true_label": true_label}

    # Zero-shot
    pred, t, tok = classify_zero_shot(complaint_text)
    row_result["zero_shot_pred"] = pred
    row_result["zero_shot_time"] = t
    row_result["zero_shot_tokens"] = tok

    # Few-shot
    pred, t, tok = classify_few_shot(complaint_text)
    row_result["few_shot_pred"] = pred
    row_result["few_shot_time"] = t
    row_result["few_shot_tokens"] = tok

    # Chain-of-Thought
    pred, expl, t, tok = classify_cot(complaint_text)
    row_result["cot_pred"] = pred
    row_result["cot_explanation"] = expl
    row_result["cot_time"] = t
    row_result["cot_tokens"] = tok

    # Self-consistency
    pred, expls, t, tok, votes = classify_self_consistency(complaint_text, n_samples=sc_samples)
    row_result["sc_pred"] = pred
    row_result["sc_explanations"] = " ||| ".join(expls)  # join multiple explanations with a separator
    row_result["sc_time"] = t
    row_result["sc_tokens"] = tok
    row_result["sc_votes"] = dict(votes)

    return row_result

## Step 1: Sanity-check on the small design set first (20 complaints)

Always test on the small set before committing to the full run — this catches bugs cheaply (20 complaints × ~7 calls each = ~140 calls, takes a couple of minutes) before you spend a full run's worth of API calls on a bug.


In [ ]:
design_results = []

for i, row in design_df.iterrows():
    print(f"[{i+1}/{len(design_df)}] Processing...")
    result = run_all_conditions(row[TEXT_COL], row["clean_category"])
    design_results.append(result)

design_results_df = pd.DataFrame(design_results)
design_results_df.to_csv("design_set_all_conditions_results.csv", index=False)
print("\nDone. Saved: design_set_all_conditions_results.csv")

# Quick accuracy check for all 4 conditions
for col in ["zero_shot_pred", "few_shot_pred", "cot_pred", "sc_pred"]:
    acc = (design_results_df[col].str.strip().str.lower() == design_results_df["true_label"].str.strip().str.lower()).mean()
    print(f"{col}: {acc:.1%} accuracy")

**Checkpoint:** look at the 4 accuracy numbers above. If they all look reasonable (not wildly different from your earlier 85% zero-shot baseline, and not near-random), you're clear to run the full test set. If something looks badly broken (e.g., one condition stuck at 0%), stop and paste me the output before continuing — don't burn the full budget on a bug.

## Step 2: The full run (175 test complaints)

This is the real experiment. It'll take a while — let it run, and don't close the Colab tab while it's going.


In [ ]:
test_results = []

for i, row in test_df.iterrows():
    print(f"[{i+1}/{len(test_df)}] True: {row['clean_category']}")
    try:
        result = run_all_conditions(row[TEXT_COL], row["clean_category"])
        test_results.append(result)
    except Exception as e:
        print(f"  ERROR on row {i}: {e}")
        time.sleep(5)  # brief pause before continuing, in case it's a rate limit
        continue

    # Save progress every 20 rows, so a crash doesn't lose everything
    if (i + 1) % 20 == 0:
        pd.DataFrame(test_results).to_csv("test_set_all_conditions_results_PARTIAL.csv", index=False)
        print(f"  --- progress saved at row {i+1} ---")

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv("test_set_all_conditions_results.csv", index=False)
print(f"\nDone. {len(test_results_df)} complaints processed. Saved: test_set_all_conditions_results.csv")

## Step 3: Download your results immediately

Don't skip this — this file is the foundation of your entire Results section.


In [ ]:
from google.colab import files
files.download("test_set_all_conditions_results.csv")
files.download("design_set_all_conditions_results.csv")

## ✅ Checkpoint — Phase 4 complete

You now have a CSV with, for every test complaint: predictions, explanations, timing, and token cost across all 4 prompting conditions.

## Next: Phase 6 — Analysis

Once this file is downloaded and safely backed up (Downloads folder + GitHub), come back and we'll compute accuracy/precision/recall/F1 per condition, build the cost-vs-accuracy chart, and do the error analysis.
